# latent-recommend M2 One-Shot Extraction

This notebook streams an MTG-Jamendo-style dataset, samples N-polar proxy playlists, loads only the ACE-Step 1.5 VAE, extracts 64-D latent embeddings, and writes the final artifact bundle for the Streamlit dashboard.

Do not use the full ACE-Step inference handler here; it loads LM/DiT components that are unnecessary for recommendation.

In [ ]:
!test -d /content/latent-recommend || git clone --depth 1 --filter=blob:none --sparse https://github.com/anuragsubedi/latent-recommend.git /content/latent-recommend
%cd /content/latent-recommend
!git sparse-checkout set latent_recommend requirements-colab.txt
!pip install -q -r requirements-colab.txt

In [ ]:
from pathlib import Path
from collections import Counter, deque
import ast
import json
import os
import random
import sqlite3
import sys
import tarfile
import tempfile
import time
import traceback

import faiss
import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "latent_recommend").exists():
    raise RuntimeError("Run the sparse-clone setup cell first so `latent_recommend/` is available.")
sys.path.insert(0, str(PROJECT_ROOT))

from latent_recommend.config import DEFAULT_TARGET_TAGS
from latent_recommend.db import connect, initialize_schema, insert_track
from latent_recommend.retrieval import normalize_embeddings
from latent_recommend.sampling import extract_record_tags, iter_n_polar_samples
from latent_recommend.vae_extraction import (
    cleanup_device,
    export_preview_mp3,
    extract_track_embedding,
    load_standalone_vae,
    process_audio_for_oobleck,
    process_hf_audio_for_oobleck,
    save_reconstruction,
)

### Add environ variables for data download and latent extraction


In [ ]:
%env SHUFFLE_SHARDS=1
%env RANDOM_SEED=42

%env PER_TAG_LIMIT=200
%env TARGET_TAGS=ambient_soundscape,electronic_dance,classical_orchestral,jazz_soul,blues_roots
# %env TARGET_TAGS=hiphop_rap,folk_acoustic,indie_rock,experimental_trip_hop,happy_pop


%env CHUNK_DURATION_SEC=200
%env MAX_AUDIO_SECONDS=200
%env CHECKPOINT_EVERY=100
%env LOG_EVERY=15
%env PREVIEW_SECONDS=15

In [ ]:
# Correct public Hugging Face dataset: https://huggingface.co/datasets/rkstgr/mtg-jamendo
DATASET_NAME = os.environ.get("MTG_JAMENDO_DATASET", "rkstgr/mtg-jamendo")
DATASET_SPLIT = os.environ.get("MTG_JAMENDO_SPLIT", "train")
TARGET_TAGS = tuple(tag.strip() for tag in os.environ.get("TARGET_TAGS", ",".join(DEFAULT_TARGET_TAGS)).split(",") if tag.strip())
PER_TAG_LIMIT = int(os.environ.get("PER_TAG_LIMIT", "200"))
CHUNK_DURATION_SEC = int(os.environ.get("CHUNK_DURATION_SEC", "300"))
SHUFFLE_SHARDS = os.environ.get("SHUFFLE_SHARDS", "1") == "1"
RANDOM_SEED = int(os.environ.get("RANDOM_SEED", "42"))

# Runtime knobs. Set MAX_AUDIO_SECONDS=0 to encode full songs.
# On H100, CHUNK_DURATION_SEC=300 with MAX_AUDIO_SECONDS=300 stayed around 24GB VRAM in tests.
MAX_AUDIO_SECONDS = int(os.environ.get("MAX_AUDIO_SECONDS", "300"))
PREVIEW_ENABLED = os.environ.get("PREVIEW_ENABLED", "1") == "1"
PREVIEW_SECONDS = int(os.environ.get("PREVIEW_SECONDS", "10"))
LOG_EVERY = int(os.environ.get("LOG_EVERY", "5"))
CHECKPOINT_EVERY = int(os.environ.get("CHECKPOINT_EVERY", "100"))
ARTIFACT_ROOT = Path(os.environ.get("ARTIFACT_ROOT", "/content/latent-recommend-artifacts"))
PREVIEW_ROOT = ARTIFACT_ROOT / "previews"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
PREVIEW_ROOT.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print({
    "dataset": DATASET_NAME,
    "split": DATASET_SPLIT,
    "target_tags": TARGET_TAGS,
    "per_tag_limit": PER_TAG_LIMIT,
    "chunk_duration_sec": CHUNK_DURATION_SEC,
    "shuffle_shards": SHUFFLE_SHARDS,
    "random_seed": RANDOM_SEED,
    "max_audio_seconds": MAX_AUDIO_SECONDS,
    "preview_enabled": PREVIEW_ENABLED,
    "preview_seconds": PREVIEW_SECONDS,
    "log_every": LOG_EVERY,
    "checkpoint_every": CHECKPOINT_EVERY,
    "artifact_root": str(ARTIFACT_ROOT),
    "device": device,
})

In [ ]:
TRACKS_URLS = {
    "train": "https://huggingface.co/datasets/rkstgr/mtg-jamendo/raw/main/train.tsv",
    "validation": "https://huggingface.co/datasets/rkstgr/mtg-jamendo/raw/main/valid.tsv",
    "valid": "https://huggingface.co/datasets/rkstgr/mtg-jamendo/raw/main/valid.tsv",
    "val": "https://huggingface.co/datasets/rkstgr/mtg-jamendo/raw/main/valid.tsv",
}

SHARD_COUNTS = {"train": 200, "validation": 22, "valid": 22, "val": 22}
SHARD_DIRS = {"train": "train", "validation": "val", "valid": "val", "val": "val"}


def parse_list(value):
    if isinstance(value, list):
        return value
    if value in (None, ""):
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []


def load_track_metadata(split):
    table = pd.read_csv(TRACKS_URLS[split], sep="\t")
    for field in ("genres", "instruments", "moods"):
        table[field] = table[field].apply(parse_list)
    return {int(row["id"]): row.to_dict() for _, row in table.iterrows()}


def iter_mtg_jamendo_tar_stream(split):
    """Stream MTG-Jamendo tar shards directly; the HF dataset script streaming path is broken."""
    tracks = load_track_metadata(split)
    shard_dir = SHARD_DIRS[split]
    shard_ids = list(range(SHARD_COUNTS[split]))
    if SHUFFLE_SHARDS:
        random.Random(RANDOM_SEED).shuffle(shard_ids)
    print(f"Streaming {len(shard_ids)} {split} shards; first shards={shard_ids[:10]}")
    for shard_id in shard_ids:
        url = f"https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/{shard_dir}/{shard_id}.tar"
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            response.raw.decode_content = True
            with tarfile.open(fileobj=response.raw, mode="r|*") as tar:
                for member in tar:
                    if not member.isfile() or not member.name.endswith(".opus"):
                        continue
                    track_id = int(Path(member.name).stem)
                    if track_id not in tracks:
                        continue
                    extracted = tar.extractfile(member)
                    if extracted is None:
                        continue
                    record = dict(tracks[track_id])
                    record["id"] = track_id
                    record["audio"] = {
                        "bytes": extracted.read(),
                        "path": member.name,
                    }
                    yield record


def waveform_from_record(record):
    audio = record.get("audio")
    if isinstance(audio, dict) and "array" in audio:
        return process_hf_audio_for_oobleck(audio, device=device)
    if isinstance(audio, dict) and "bytes" in audio:
        with tempfile.NamedTemporaryFile(suffix=".opus") as audio_file:
            audio_file.write(audio["bytes"])
            audio_file.flush()
            return process_audio_for_oobleck(audio_file.name, device=device)
    raise KeyError("Could not find usable audio in record.")


def crop_waveform(waveform, max_seconds, sr=48000):
    """Center-crop long tracks for faster experiments; max_seconds=0 keeps full songs."""
    if max_seconds <= 0:
        return waveform
    max_samples = int(max_seconds * sr)
    if waveform.shape[-1] <= max_samples:
        return waveform
    start = (waveform.shape[-1] - max_samples) // 2
    return waveform[:, :, start : start + max_samples].contiguous()


def text_value(record, keys, default=None):
    for key in keys:
        value = record.get(key)
        if value not in (None, ""):
            return value
    return default


def metadata_row(faiss_id, primary_tag, record, preview_path):
    tags = sorted(extract_record_tags(record) | {primary_tag})
    return {
        "faiss_id": faiss_id,
        "track_id": str(text_value(record, ["track_id", "id", "path"], f"track-{faiss_id}")),
        "title": str(text_value(record, ["title", "track_title", "name"], f"Track {faiss_id}")),
        "artist": str(text_value(record, ["artist", "artist_name", "artist_id"], "Unknown artist")),
        "album": text_value(record, ["album", "album_title", "album_id"], None),
        "duration": text_value(record, ["duration", "duration_seconds", "durationInSec", "duration_in_sec"], None),
        "primary_tag": primary_tag,
        "tags": ",".join(tags),
        "audio_url": text_value(record, ["audio_url", "download_url", "url"], None),
        "preview_path": str(preview_path.relative_to(ARTIFACT_ROOT)) if preview_path else None,
        "split": "eval",
    }

## Load the VAE once

The VAE is the only ACE-Step component needed for this recommender. Keep it in float32 to match the initial smoke-test notebook and avoid 1D convolution underflow issues.

In [ ]:
vae = load_standalone_vae(device=device)

## Stream, extract, and write artifacts

In [ ]:
db_path = ARTIFACT_ROOT / "metadata.db"
index_path = ARTIFACT_ROOT / "vectors.index"
embeddings_path = ARTIFACT_ROOT / "embeddings.npy"
manifest_path = ARTIFACT_ROOT / "manifest.json"
checkpoint_manifest_path = ARTIFACT_ROOT / "checkpoint_manifest.json"
runtime_profile_path = ARTIFACT_ROOT / "runtime_profile.jsonl"

conn = connect(db_path)
initialize_schema(conn)
index = faiss.IndexFlatIP(64)
embeddings = []
errors = []
runtime_logs = []
tag_counts = Counter()
recent_seconds = deque(maxlen=max(LOG_EVERY, 1))
run_started = time.perf_counter()


def write_checkpoint(reason):
    if not embeddings:
        return
    faiss.write_index(index, str(index_path))
    np.save(embeddings_path, np.vstack(embeddings).astype("float32"))
    checkpoint_manifest_path.write_text(json.dumps({
        "reason": reason,
        "dataset": DATASET_NAME,
        "split": DATASET_SPLIT,
        "target_tags": TARGET_TAGS,
        "per_tag_limit": PER_TAG_LIMIT,
        "chunk_duration_sec": CHUNK_DURATION_SEC,
        "shuffle_shards": SHUFFLE_SHARDS,
        "random_seed": RANDOM_SEED,
        "max_audio_seconds": MAX_AUDIO_SECONDS,
        "preview_enabled": PREVIEW_ENABLED,
        "preview_seconds": PREVIEW_SECONDS,
        "tracks_written": len(embeddings),
        "tag_counts": dict(tag_counts),
        "runtime_profile_path": str(runtime_profile_path),
        "runtime_log_records": len(runtime_logs),
        "errors": errors,
    }, indent=2))
    print(f"Checkpoint saved after {len(embeddings)} tracks -> {ARTIFACT_ROOT}")

stream = iter_mtg_jamendo_tar_stream(DATASET_SPLIT)
sample_iter = iter_n_polar_samples(stream, TARGET_TAGS, per_tag_limit=PER_TAG_LIMIT)
total_target = PER_TAG_LIMIT * len(TARGET_TAGS)
progress = tqdm(sample_iter, total=total_target, desc="Embedding tracks")

for _, (primary_tag, record) in enumerate(progress):
    faiss_id = len(embeddings)
    timings = {}
    track_started = time.perf_counter()
    try:
        step_started = time.perf_counter()
        waveform = waveform_from_record(record)
        timings["decode"] = time.perf_counter() - step_started

        original_seconds = waveform.shape[-1] / 48000
        waveform = crop_waveform(waveform, MAX_AUDIO_SECONDS)
        processed_seconds = waveform.shape[-1] / 48000

        step_started = time.perf_counter()
        embedding = extract_track_embedding(
            vae,
            waveform,
            chunk_duration_sec=CHUNK_DURATION_SEC,
        )
        timings["encode"] = time.perf_counter() - step_started

        step_started = time.perf_counter()
        normalized = normalize_embeddings(embedding.reshape(1, -1)).astype("float32")
        index.add(normalized)
        embeddings.append(normalized.squeeze(0))
        timings["index"] = time.perf_counter() - step_started

        preview_path = None
        if PREVIEW_ENABLED:
            preview_path = PREVIEW_ROOT / f"{faiss_id:06d}.mp3"
            step_started = time.perf_counter()
            try:
                export_preview_mp3(waveform, preview_path, duration_sec=PREVIEW_SECONDS)
            except Exception as preview_error:
                print(f"Preview failed for {faiss_id}: {preview_error}")
                preview_path = None
            timings["preview"] = time.perf_counter() - step_started
        else:
            timings["preview"] = 0.0

        step_started = time.perf_counter()
        insert_track(conn, metadata_row(faiss_id, primary_tag, record, preview_path))
        conn.commit()
        timings["db"] = time.perf_counter() - step_started

        tag_counts[primary_tag] += 1
        track_seconds = time.perf_counter() - track_started
        recent_seconds.append(track_seconds)
        progress.set_postfix({
            "ok": len(embeddings),
            "tag": primary_tag,
            "sec/track": f"{np.mean(recent_seconds):.1f}",
            "audio_s": f"{processed_seconds:.0f}/{original_seconds:.0f}",
        })

        elapsed = time.perf_counter() - run_started
        remaining = max(total_target - len(embeddings), 0)
        eta_hours = (remaining * np.mean(recent_seconds) / 3600) if recent_seconds else 0
        log_record = {
            "n": len(embeddings),
            "total_target": total_target,
            "track_id": str(record.get("id")),
            "tag": primary_tag,
            "original_seconds": original_seconds,
            "processed_seconds": processed_seconds,
            "total_seconds": track_seconds,
            "decode_seconds": timings["decode"],
            "encode_seconds": timings["encode"],
            "preview_seconds": timings["preview"],
            "db_seconds": timings["db"],
            "rolling_avg_seconds": float(np.mean(recent_seconds)),
            "elapsed_minutes": elapsed / 60,
            "eta_hours": eta_hours,
            "tag_counts": dict(tag_counts),
        }
        runtime_logs.append(log_record)
        with runtime_profile_path.open("a") as profile_file:
            profile_file.write(json.dumps(log_record) + "\n")

        if len(embeddings) == 1 or len(embeddings) % LOG_EVERY == 0:
            print(
                f"[{len(embeddings)}/{total_target}] "
                f"track_id={record.get('id')} tag={primary_tag} "
                f"audio={processed_seconds:.1f}s/{original_seconds:.1f}s "
                f"total={track_seconds:.2f}s decode={timings['decode']:.2f}s "
                f"encode={timings['encode']:.2f}s preview={timings['preview']:.2f}s "
                f"db={timings['db']:.2f}s avg={np.mean(recent_seconds):.2f}s "
                f"elapsed={elapsed/60:.1f}m eta={eta_hours:.1f}h counts={dict(tag_counts)}"
            )

        if CHECKPOINT_EVERY > 0 and len(embeddings) % CHECKPOINT_EVERY == 0:
            write_checkpoint(reason="periodic")

        del waveform
    except Exception as exc:
        errors.append({
            "candidate_faiss_id": faiss_id,
            "track_id": str(record.get("id", "unknown")),
            "error": repr(exc),
        })
        traceback.print_exc(limit=2)
    finally:
        cleanup_device()

if not embeddings:
    raise RuntimeError("No embeddings were written. Check target tags and dataset streaming logs.")

write_checkpoint(reason="final")
manifest_path.write_text(json.dumps({
    "dataset": DATASET_NAME,
    "split": DATASET_SPLIT,
    "target_tags": TARGET_TAGS,
    "per_tag_limit": PER_TAG_LIMIT,
    "chunk_duration_sec": CHUNK_DURATION_SEC,
    "shuffle_shards": SHUFFLE_SHARDS,
    "random_seed": RANDOM_SEED,
    "max_audio_seconds": MAX_AUDIO_SECONDS,
    "preview_enabled": PREVIEW_ENABLED,
    "preview_seconds": PREVIEW_SECONDS,
    "checkpoint_every": CHECKPOINT_EVERY,
    "tracks_written": len(embeddings),
    "tag_counts": dict(tag_counts),
    "runtime_profile_path": str(runtime_profile_path),
    "runtime_log_records": len(runtime_logs),
    "errors": errors,
}, indent=2))
conn.close()

print(f"Wrote {len(embeddings)} embeddings to {ARTIFACT_ROOT}")

## Next step

After downloading the generated `artifacts/` bundle locally, run:

```bash
python scripts/evaluate_artifacts.py
python scripts/validate_artifacts.py
streamlit run app/streamlit_app.py
```